In [1]:
%load_ext autoreload
%autoreload all

In [36]:
# Standard Library imports
from pathlib import Path

# External imports
import fiftyone as fo
import fiftyone.brain as fob
from fiftyone.core.dataset import load_dataset, Dataset
from fiftyone import ViewField as F

# Local imports
from fiftyone_tools import (
    add_samples_to_dataset,
    find_duplicates_near,
    transform_tag_into_label,
    export_yolo_classification,
    remove_tags,
    split_dataset_train_val,
    select_with_classification
)

In [3]:
dataset_name = "masked_fish_patches"
try:
    dataset = Dataset(dataset_name, persistent=True, overwrite=False)
except ValueError:
    print("error")
    dataset = load_dataset(dataset_name, create_if_necessary=False)

You are running the oldest supported major version of MongoDB. Please refer to https://deprecation.voxel51.com for deprecation notices. You can suppress this exception by setting your `database_validation` config parameter to `False`. See https://docs.voxel51.com/user_guide/config.html#configuring-a-mongodb-connection for more information


10/24/2025 11:51:21 - WARNING - fiftyone.core.odm.database -   You are running the oldest supported major version of MongoDB. Please refer to https://deprecation.voxel51.com for deprecation notices. You can suppress this exception by setting your `database_validation` config parameter to `False`. See https://docs.voxel51.com/user_guide/config.html#configuring-a-mongodb-connection for more information


In [ ]:
ROOT_PATH = Path(r"F:\DATASETS\GIL LAB\Example SAM2\stationary\GX056267_frames2")
sources = ROOT_PATH / "output_patches"
add_samples_to_dataset(dataset, sources)

 100% |███████████████| 1594/1594 [453.2ms elapsed, 0s remaining, 3.5K samples/s]      


10/24/2025 11:53:44 - INFO - eta.core.utils -    100% |███████████████| 1594/1594 [453.2ms elapsed, 0s remaining, 3.5K samples/s]      


In [5]:
# When auto=False is provided, a new App window is created only when you call session.show():
session = fo.launch_app(dataset, browser="chrome", auto=False)

# Now open http://localhost:5151


Could not connect session, trying again in 10 seconds

Session launched. Run `session.show()` to open the App in a cell output.


10/24/2025 11:51:38 - INFO - fiftyone.core.session.session -   Session launched. Run `session.show()` to open the App in a cell output.


Find near duplicates

In [ ]:
# Near duplicates
# "dinov2-vitb14-torch": 330 MB, normal results
# "dinov2-vitl14-reg-torch": 1GB
# "clip-vit-base32-torch": 2.6 GB
brain_key_similarity = "similarity_clip"
model_name = "clip-vit-base32-torch"
similarity_index = find_duplicates_near(
    dataset, model_name, brain_key_similarity, thresh=0.4
)

Computing embeddings...


10/24/2025 11:54:16 - INFO - fiftyone.brain.internal.core.utils -   Computing embeddings...


 100% |███████████████| 1594/1594 [1.1m elapsed, 0s remaining, 26.8 samples/s]      


10/24/2025 11:55:21 - INFO - eta.core.utils -    100% |███████████████| 1594/1594 [1.1m elapsed, 0s remaining, 26.8 samples/s]      


Computing duplicate samples...


10/24/2025 11:55:23 - INFO - fiftyone.brain.similarity -   Computing duplicate samples...


Duplicates computation complete


10/24/2025 11:55:23 - INFO - fiftyone.brain.similarity -   Duplicates computation complete


Index type: sklearn
Index created with model: clip-vit-base32-torch
Index size: 1594
Index supports prompts: True


In [38]:
# Use the computed near duplicates index with a different threshold to search again for duplicates
similarity_index.find_duplicates(thresh=0.4)  # Higher thresh, more images shown
session.view = similarity_index.duplicates_view()

Computing duplicate samples...


10/24/2025 12:19:09 - INFO - fiftyone.brain.similarity -   Computing duplicate samples...


Duplicates computation complete


10/24/2025 12:19:09 - INFO - fiftyone.brain.similarity -   Duplicates computation complete


Show unique

In [29]:
similarity_index.find_duplicates(thresh=0.4)  # Higher thresh, less images shown
session.view = similarity_index.unique_view()

Computing duplicate samples...


10/24/2025 12:00:03 - INFO - fiftyone.brain.similarity -   Computing duplicate samples...


Duplicates computation complete


10/24/2025 12:00:03 - INFO - fiftyone.brain.similarity -   Duplicates computation complete


In [13]:
# Generate a 2D visualization of near duplicates
viz_results = fob.compute_visualization(dataset, brain_key="visualization", similarity_index=similarity_index)

Retrieving embeddings from similarity index...


10/24/2025 11:56:26 - INFO - fiftyone.brain.internal.core.utils -   Retrieving embeddings from similarity index...


Generating visualization...


10/24/2025 11:56:26 - INFO - fiftyone.brain.visualization -   Generating visualization...
c:\Users\Manuel\miniforge3\envs\cv_env\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


UMAP( verbose=True)
Fri Oct 24 11:56:34 2025 Construct fuzzy simplicial set
Fri Oct 24 11:56:37 2025 Finding Nearest Neighbors
Fri Oct 24 11:56:46 2025 Finished Nearest Neighbor Search
Fri Oct 24 11:56:50 2025 Construct embedding


Epochs completed:   0%|            0/500 [00:00]

	completed  0  /  500 epochs
	completed  50  /  500 epochs
	completed  100  /  500 epochs
	completed  150  /  500 epochs
	completed  200  /  500 epochs
	completed  250  /  500 epochs
	completed  300  /  500 epochs
	completed  350  /  500 epochs
	completed  400  /  500 epochs
	completed  450  /  500 epochs
Fri Oct 24 11:56:53 2025 Finished embedding


In [28]:
plot = similarity_index.visualize_unique(viz_results, backend="plotly") 
plot.show()

FigureWidget({
    'data': [{'customdata': array(['68fbaf183a564b0723995258', '68fbaf183a564b072399525b',
                                   '68fbaf183a564b0723995261', ..., '68fbaf183a564b072399588d',
                                   '68fbaf183a564b072399588e', '68fbaf183a564b072399588f'], dtype=object),
              'hovertemplate': ('<b>label: %{text}</b><br>x, y ' ... ': %{customdata}<extra></extra>'),
              'line': {'color': '#3366CC'},
              'mode': 'markers',
              'name': 'other',
              'showlegend': True,
              'text': array(['other', 'other', 'other', ..., 'other', 'other', 'other'], dtype=object),
              'type': 'scattergl',
              'uid': 'bd501458-9ccc-43b0-aa19-ae284ebb551b',
              'x': {'bdata': ('2Fz+v31lCcDdKJq/9yekv9oBPUAy+T' ... 'HAzbhlwM8DA8CjJEDAl1BmwDkwZ0E='),
                    'dtype': 'f4'},
              'y': {'bdata': ('UfgGwKZuB8CjS2tA3TRlQP/yfj/TVY' ... 'zAQnoUwEdSXcArsda/0oBkwPKoEkA='),
     

In [27]:
# Unique examples are to the right of the threshold bar
distances_plot = similarity_index.plot_distances(backend="plotly")
distances_plot.show()

FigureWidget({
    'data': [{'customdata': {'bdata': ('wLn8zu/DmD9M6u4XQ6aeP0zq7hdDpp' ... 'hR+uI//islKFH64j+CvWzCYynjPw=='),
                             'dtype': 'f8',
                             'shape': '100, 2'},
              'hovertemplate': ('<b>count: %{y}</b><br>distance' ... 'omdata[1]:.2f}]<extra></extra>'),
              'marker': {'color': '#FF6D04'},
              'offset': 0,
              'showlegend': False,
              'type': 'bar',
              'uid': '34368ba8-88c8-41cd-9793-ac4fbd77db42',
              'width': {'bdata': ('MMLII02Jdz8wwsgjTYl3PyjCyCNNiX' ... 'gjTYl3P4DCyCNNiXc/AMLII02Jdz8='),
                        'dtype': 'f8'},
              'x': {'bdata': ('wLn8zu/DmD9M6u4XQ6aeP2yNcDBLRK' ... 'bzK5ziP3ma3Y0+y+I//islKFH64j8='),
                    'dtype': 'f8'},
              'y': {'bdata': ('CwgPDwoRCAQJBQgFAwgFBBIJEA0JEB' ... 'kEAwUCAAEAAQIBAAACAAAAAAAAAQ=='),
                    'dtype': 'i1'}},
             {'hovertemplate': '<b>thresh: %{x}</b><extr

In [40]:
transform_tag_into_label(dataset, tag="correct_fish_mask", label="correct_fish_mask")
transform_tag_into_label(dataset, tag="other", label="other")

Export to Ultralytics YOLO classification format

In [41]:
export_path = ROOT_PATH/"yolo_classification_dataset"
train_split = 0.7
remove_tags(dataset, ["train", "val", "test"])
#Select only samples with Ground Truth
view = select_with_classification(dataset)
split_dataset_train_val(view, train_split=train_split)
export_yolo_classification(view, export_path=export_path, splits=["train", "val"])

train: 435
val: 186
Exporting 'train' data to 'F:\DATASETS\GIL LAB\Example SAM2\stationary\GX056267_frames2\yolo_classification_dataset\train'
Directory 'F:\DATASETS\GIL LAB\Example SAM2\stationary\GX056267_frames2\yolo_classification_dataset\train' already exists; export will be merged with existing files


10/24/2025 12:21:15 - WARNING - fiftyone.core.collections -   Directory 'F:\DATASETS\GIL LAB\Example SAM2\stationary\GX056267_frames2\yolo_classification_dataset\train' already exists; export will be merged with existing files


 100% |█████████████████| 435/435 [1.3s elapsed, 0s remaining, 328.0 samples/s]         


10/24/2025 12:21:16 - INFO - eta.core.utils -    100% |█████████████████| 435/435 [1.3s elapsed, 0s remaining, 328.0 samples/s]         


Exporting 'val' data to 'F:\DATASETS\GIL LAB\Example SAM2\stationary\GX056267_frames2\yolo_classification_dataset\val'
Directory 'F:\DATASETS\GIL LAB\Example SAM2\stationary\GX056267_frames2\yolo_classification_dataset\val' already exists; export will be merged with existing files


10/24/2025 12:21:16 - WARNING - fiftyone.core.collections -   Directory 'F:\DATASETS\GIL LAB\Example SAM2\stationary\GX056267_frames2\yolo_classification_dataset\val' already exists; export will be merged with existing files


 100% |█████████████████| 186/186 [667.4ms elapsed, 0s remaining, 278.7 samples/s]      


10/24/2025 12:21:17 - INFO - eta.core.utils -    100% |█████████████████| 186/186 [667.4ms elapsed, 0s remaining, 278.7 samples/s]      
